In [0]:
# Runs as the pseudonymization SP. Creates Unity Catalog secrets (keys never leave this job, never printed)
# and UC Python UDFs that declare those secrets in a SECRETS clause, so callers never see key material.
dbutils.widgets.text("catalog", "serverless_stable_yit814_catalog")
catalog = dbutils.widgets.get("catalog")
VAULT, GOV = f"{catalog}.pii_vault", f"{catalog}.pii_governance"
KEYS = ["ssn_key_v1", "phone_key_v1", "dob_key_v1", "hmac_key_v1"]

In [0]:
import base64, os
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound

w = WorkspaceClient()
for key in KEYS:
    full_name = f"{VAULT}.{key}"
    try:
        w.api_client.do("GET", f"/api/2.1/unity-catalog/secrets/{full_name}")
        print(f"exists : {full_name} (kept, not rotated)")
    except NotFound:
        w.api_client.do("POST", "/api/2.1/unity-catalog/secrets", body={
            "catalog_name": catalog, "schema_name": "pii_vault", "name": key,
            # 32 random bytes = AES-256 key, stored base64-encoded. The value is never printed or logged.
            "value": base64.b64encode(os.urandom(32)).decode(),
            "comment": "PII demo key material. READ SECRET only for the pseudonymization service principal.",
        })
        print(f"created: {full_name}")

In [0]:
enc_secrets = ", ".join(f"{VAULT}.{k}" for k in KEYS if k != "hmac_key_v1")
env = "ENVIRONMENT (dependencies = '[\"cryptography\"]', environment_version = '6')"

# Ciphertext layout = base64(12-byte IV || ciphertext || 16-byte GCM tag), the same layout as the built-in
# aes_encrypt(..., 'GCM'), so try_aes_decrypt can also decrypt it given the key.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {VAULT}.pii_encrypt(value STRING, key_name STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Pattern A: AES-256-GCM field encryption. Random IV, so not joinable; use pii_hmac for joins.'
SECRETS ({enc_secrets})
{env}
AS $$
import base64, os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from databricks.secrets import get
if value is None:
    return None
if key_name not in ('ssn_key_v1', 'phone_key_v1', 'dob_key_v1'):
    raise ValueError('unknown key name')
key = base64.b64decode(get(catalog='{catalog}', schema='pii_vault', key=key_name))
nonce = os.urandom(12)
return base64.b64encode(nonce + AESGCM(key).encrypt(nonce, value.encode('utf-8'), None)).decode('ascii')
$$
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {VAULT}.pii_decrypt(ciphertext STRING, key_name STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Re-identification. EXECUTE only for the pseudonymization service principal. Returns NULL on wrong key or corrupt data.'
SECRETS ({enc_secrets})
{env}
AS $$
import base64
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from databricks.secrets import get
if ciphertext is None:
    return None
if key_name not in ('ssn_key_v1', 'phone_key_v1', 'dob_key_v1'):
    raise ValueError('unknown key name')
key = base64.b64decode(get(catalog='{catalog}', schema='pii_vault', key=key_name))
raw = base64.b64decode(ciphertext)
try:
    return AESGCM(key).decrypt(raw[:12], raw[12:], None).decode('utf-8')
except Exception:
    return None
$$
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {VAULT}.pii_hmac(value STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Keyed hash (HMAC-SHA256) next to the ciphertext so encrypted columns stay joinable / countable.'
SECRETS ({VAULT}.hmac_key_v1)
ENVIRONMENT (environment_version = '6')
AS $$
import base64, hashlib, hmac
from databricks.secrets import get
if value is None:
    return None
key = base64.b64decode(get(catalog='{catalog}', schema='pii_vault', key='hmac_key_v1'))
return hmac.new(key, value.encode('utf-8'), hashlib.sha256).hexdigest()
$$
""")

# One call per protected column: ciphertext + keyed hash together. Queries are limited to 5 UC Python UDF
# calls (public preview), so the pipeline uses this combined UDF instead of pii_encrypt + pii_hmac per column.
all_secrets = ", ".join(f"{VAULT}.{k}" for k in KEYS)
spark.sql(f"""
CREATE OR REPLACE FUNCTION {VAULT}.pii_protect(value STRING, key_name STRING)
RETURNS STRUCT<enc: STRING, hmac: STRING>
LANGUAGE PYTHON
COMMENT 'Pattern A in one call: AES-256-GCM ciphertext (enc) + HMAC-SHA256 (hmac) of the same value.'
SECRETS ({all_secrets})
{env}
AS $$
import base64, hashlib, hmac, os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from databricks.secrets import get
if value is None:
    return None
if key_name not in ('ssn_key_v1', 'phone_key_v1', 'dob_key_v1'):
    raise ValueError('unknown key name')
key = base64.b64decode(get(catalog='{catalog}', schema='pii_vault', key=key_name))
hkey = base64.b64decode(get(catalog='{catalog}', schema='pii_vault', key='hmac_key_v1'))
nonce = os.urandom(12)
enc = base64.b64encode(nonce + AESGCM(key).encrypt(nonce, value.encode('utf-8'), None)).decode('ascii')
return {{'enc': enc, 'hmac': hmac.new(hkey, value.encode('utf-8'), hashlib.sha256).hexdigest()}}
$$
""")

# Row filter used by the secure-by-default ABAC policies: nobody targeted by the policy sees any row.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {GOV}.pii_deny_all()
RETURNS BOOLEAN
COMMENT 'ABAC row filter for restricted zones: returns false for every row'
RETURN false
""")
print("UDFs created")

In [0]:
test = spark.sql(f"""
WITH t AS (SELECT '123456789' AS v)
SELECT
  {VAULT}.pii_decrypt({VAULT}.pii_encrypt(v, 'ssn_key_v1'), 'ssn_key_v1') = v AS roundtrip_ok,
  {VAULT}.pii_encrypt(v, 'ssn_key_v1') <> {VAULT}.pii_encrypt(v, 'ssn_key_v1') AS random_iv_ok,
  {VAULT}.pii_hmac(v) = {VAULT}.pii_hmac(v) AS hmac_deterministic_ok,
  {VAULT}.pii_decrypt({VAULT}.pii_encrypt(v, 'ssn_key_v1'), 'phone_key_v1') IS NULL AS wrong_key_returns_null
FROM t
""").first()
print(test.asDict())
assert all(test.asDict().values()), "UDF self-test failed"

t2 = spark.sql(f"""
SELECT {VAULT}.pii_decrypt(p.enc, 'ssn_key_v1') = '123456789' AS protect_roundtrip_ok,
       p.hmac = {VAULT}.pii_hmac('123456789')               AS protect_hmac_ok
FROM (SELECT {VAULT}.pii_protect('123456789', 'ssn_key_v1') AS p)
""").first().asDict()
print(t2)
assert all(t2.values()), "pii_protect self-test failed"